# Reproduce Table 3 via Quantum Reduction Pipeline

This notebook reproduces the `sl_3` boundary example at $k=-9/4$ with $f=[2,1]$:
- Enumerate boundary admissible AKM weights by $w(\hat\Pi_u)\subset\hat\Delta_+$.
- Apply DS non-vanishing filter (for this case: $\lambda_0 \neq 0$).
- Pair surviving modules by the involution used in the note.

In [ ]:
from fractions import Fraction
from pyw.core.affine_lie_algebra import AffineLieAlgebra
from pyw.utils.predicates import is_positive_affine_root

alg = AffineLieAlgebra(['A', 2, 1])
Wext = alg.extended_affine_weyl_group()
k = Fraction(-9, 4)
u = 4
Lam0 = alg.fundamental_weights()[0]
rho = alg.affine_rho()
Pi_u = [u * alg.delta() - alg.affine_theta(), alg.affine_simple_roots()[1], alg.affine_simple_roots()[2]]

def to_frac(x):
    if isinstance(x, Fraction):
        return x
    if hasattr(x, 'numerator') and hasattr(x, 'denominator'):
        n = x.numerator() if callable(x.numerator) else x.numerator
        d = x.denominator() if callable(x.denominator) else x.denominator
        return Fraction(int(n), int(d))
    return Fraction(x)

def affine_dynkin_labels(weight):
    mc = weight.finite_part.monomial_coefficients()
    l1 = to_frac(mc.get(1, 0))
    l2 = to_frac(mc.get(2, 0))
    l0 = to_frac(weight.level) - l1 - l2
    return (l0, l1, l2)

In [ ]:
# 1) Enumerate AKM boundary admissible weights
elements = Wext.elements_as_semi_direct_product(translation_bounds={1: (-4, 4), 2: (-4, 4)})
by_label = {}
for w in elements:
    if not all(is_positive_affine_root(w.action(a)) for a in Pi_u):
        continue
    Lam = w.action(k * Lam0 + rho) - rho
    lbl = affine_dynkin_labels(Lam)
    coeff = w.translation_vector.monomial_coefficients()
    metric = (sum(abs(int(v)) for v in coeff.values()), len(w.reduced_word()))
    if lbl not in by_label or metric < by_label[lbl][0]:
        by_label[lbl] = (metric, w)

all16 = sorted(by_label.keys())
print('AKM boundary admissible count =', len(all16))
for x in all16:
    print(x)

In [ ]:
# 2) DS non-vanishing filter for this case: lambda0 != 0
surviving12 = [x for x in all16 if x[0] != 0]
print('Surviving count =', len(surviving12))

# 3) Pairing involution used in the note
c = -(k + 1)
surv_set = set(surviving12)
pairs = []
used = set()
for lbl in sorted(surviving12):
    if lbl in used:
        continue
    _, a, b = lbl
    p1 = -c - b
    p2 = -c - a
    partner = (k - p1 - p2, p1, p2)
    if partner in surv_set:
        pair = tuple(sorted([lbl, partner]))
        if pair not in pairs:
            pairs.append(pair)
            used.add(lbl)
            used.add(partner)

print('Independent pair count =', len(pairs))
for p in sorted(pairs):
    print(p)